# Incubator State Estimation with a Kalman Filter

Next we contruct the kalman filter, using the system sympy library to transform the system symbolic equations to a discrete time system.
We adjust the equations from [ModellingIncubatorDynamics](../3-Physics-Modelling/2-ModellingIncubatorDynamics.ipynb).

In [ ]:
%pip install control
%pip install plotly
%pip install filterpy

In [ ]:
import sympy as sp
import numpy as np
from control import ss

# Parameters
C_air = sp.Symbol("C_air", real=True)  # Specific heat capacity
G_box = sp.Symbol("G_box", real=True)  # Specific heat capacity
C_heater = sp.Symbol("C_heater", real=True)  # Specific heat capacity
G_heater = sp.Symbol("G_heater", real=True)  # Specific heat capacity

# Constants
V_heater = sp.Symbol("V_heater", real=True)
i_heater = sp.Symbol("i_heater", real=True)

# Inputs
in_room_temperature = sp.Symbol("T_room", real=True)
on_heater = sp.Symbol("on_heater", real=True)

# States
T = sp.Symbol("T", real=True)
T_heater = sp.Symbol("T_h", real=True)

power_in = on_heater * V_heater * i_heater

power_transfer_heat = G_heater * (T_heater - T)

total_power_heater = power_in - power_transfer_heat

power_out_box = G_box * (T - in_room_temperature)

total_power_box = power_transfer_heat - power_out_box

der_T = (1.0 / C_air) * (total_power_box)
der_T_heater = (1.0 / C_heater) * (total_power_heater)

# Turn above into a continuous time linear system
"""
States are:
[[ T_heater ]
[ T        ]]

Inputs are: 
[ [ on_heater ], 
[ in_room_temperature ]]
"""

# We use the derivative (diff) as a shortcut to get the terms 
#  that need to be multiplied by each state in A*x
A = sp.Matrix([
    [der_T_heater.diff(T_heater),   der_T_heater.diff(T)],
    [der_T.diff(T_heater),          der_T.diff(T)       ]
])

B = sp.Matrix([
    [der_T_heater.diff(on_heater),  der_T_heater.diff(in_room_temperature)  ],
    [der_T.diff(on_heater),         der_T.diff(in_room_temperature)         ]
])

# Observation matrix: only T can be measured, not T_heater.
C = sp.Matrix([[0.0, 1.0]])

In [ ]:
print(der_T)

In [ ]:
print(der_T_heater)

In [ ]:
A

In [ ]:
B

In [ ]:
C

In [ ]:
# System parameters
V_heater_num = HEATER_VOLTAGE = 12.0
i_heater_num = HEATER_CURRENT = 10.45
C_air_num = 68.21
G_box_num = 0.74 
C_heater_num = 243.46
G_heater_num = 0.87

# Replace constants and get numerical matrices
def replace_constants(m):
    return np.array(m.subs({
        V_heater: V_heater_num,
        i_heater: i_heater_num,
        C_air: C_air_num,
        G_box: G_box_num,
        C_heater: C_heater_num,
        G_heater: G_heater_num
    })).astype(np.float64)


In [ ]:

# Numerical simulation parameters
STEP_SIZE = 3.0

# Numerical matrices
A_num = replace_constants(A)

B_num = replace_constants(B)

C_num = replace_constants(C)

D_num = np.array([[0.0, 0.0]])

ct_system = ss(A_num, B_num, C_num, D_num)
dt_system = ct_system.sample(STEP_SIZE, method="backward_diff")

dt_system

Now we have a discrete time linear system, we can construct the kalman filter.

In [ ]:
from filterpy.common import Q_discrete_white_noise
from filterpy.kalman import KalmanFilter

initial_heat_temperature = 25.0
initial_box_temperature = 25.0

std_dev_measurement_noise = 0.9
std_dev_process_noise = 0.1

# This is the initial uncertainty (covariance matrix)
process_covariance_init = np.array([[100., 0.],
                                    [0., 100.]])

f = KalmanFilter(dim_x=2, dim_z=1, dim_u=2)
f.x = np.array([[initial_heat_temperature],  # T_heater at t=0
                [initial_box_temperature]])  # T at t=0
f.F = dt_system.A
f.B = dt_system.B
f.H = dt_system.C
f.P = process_covariance_init
f.R = np.array([[std_dev_measurement_noise]])
f.Q = Q_discrete_white_noise(dim=2, dt=STEP_SIZE, var=std_dev_process_noise ** 2)

f

In [ ]:
from data_handling import load_data
import pandas
import math

# Load the data
time_unit = 'ns'
data, _ = load_data("./lid_opening_experiment_jan_2021/lid_opening_experiment_jan_2021.csv",
                    HEATER_VOLTAGE, HEATER_CURRENT,
                    desired_timeframe=(- math.inf, math.inf),
                    time_unit=time_unit,
                    normalize_time=False,
                    convert_to_seconds=True)
events = pandas.read_csv("./lid_opening_experiment_jan_2021/events.csv")
events["timestamp_ns"] = pandas.to_datetime(events["time"], unit=time_unit)

data

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, 1)

ax1.plot(data["time"], data["t1"], label="t1")
ax1.plot(data["time"], data["t2"], label="t2")
ax1.plot(data["time"], data["t3"], label="t3")
ax1.plot(data["time"], data["average_temperature"], label="average_temperature")
ax1.legend()

ax2.plot(data["time"], data["heater_on"], label="heater_on")
ax2.plot(data["time"], data["fan_on"], label="fan_on")
ax2.legend()

ax3.plot(data["time"], data["execution_interval"], label="execution_interval")
ax3.plot(data["time"], data["elapsed"], label="elapsed")
ax3.legend()

ax4.plot(data["time"], data["power_in"], label="power_in")
ax4.legend()

plt.show()

In [ ]:
events

In [ ]:
# Inputs to _plant
measurements_heater = np.array([1.0 if b else 0.0 for b in data["heater_on"]])
measurements_Troom = data["t1"].to_numpy()

# System state measurements (partial)
measurements_T = data["average_temperature"].to_numpy()

kalman_prediction = []
kalman_prediction_prior_T = []
kalman_process_noise_Theater = []
kalman_process_noise_T = []
kalman_process_noise_Theater_prior = []
kalman_process_noise_T_prior = []
for i in range(len(measurements_heater)):
    f.predict(u=np.array([
            [measurements_heater[i]],
            [measurements_Troom[i]]
        ]))
    f.update(np.array([[measurements_T[i]]]))
    x = f.x
    kalman_process_noise_Theater_prior.append(np.sqrt(f.P_prior[0,0]))
    kalman_process_noise_T_prior.append(np.sqrt(f.P_prior[1,1]))
    kalman_process_noise_Theater.append(np.sqrt(f.P[0,0]))
    kalman_process_noise_T.append(np.sqrt(f.P[1,1]))
    kalman_prediction_prior_T.append(f.x_prior[1][0])
    kalman_prediction.append(x)

kalman_prediction = np.array(kalman_prediction).squeeze(2)
kalman_prediction


In [ ]:
from data_handling import plotly_incubator_data

fig = plotly_incubator_data(data,
                            compare_to={
                                "Kalman": {
                                    "timestamp_ns": data["timestamp_ns"],
                                    "T": kalman_prediction[:, 1],
                                    "T_heater": kalman_prediction[:, 0]
                                },
                            },
                            heater_T_data={
                                "Kalman": {
                                    "timestamp_ns": data["timestamp_ns"],
                                    "T_heater": kalman_prediction[:, 0]
                                },
                            },
                            events=events,
                            overlay_heater=True,
                            show_hr_time=True)

# Save plotly interactive plot
import plotly.io as pio
pio.write_html(fig, file="incubator_KF.html")

fig

## Advanced Exercises

1. Create a service similar to the [NNBasedMonitoringService](../3-Physics-Modelling/5-NNBasedMonitoringService.ipynb) that uses a kalman filter instead. Essentially, each time a new temperature sample is available, the KF service should output a predicted temperature, predicted heater temperature, prediction error, and any other value you would like to output. Deploy it and test it with opening the lid. Note that you will have to adjust the physical system parameters above, to be the same as the physical system parameters used in the [PT Emulator](../5-IncubatorPTEmulator/1-IncubatorPTEmulator.ipynb), since the above dataset was from an earlier version of the incubator.
